# RF_Surrogate_Room_Generalization

Historical experiment source from the internship project. Outputs and personal runtime metadata are cleared. This version is retained for provenance; see the root README for the maintained entry points. Colab Drive paths describe the original environment and require your own matching inputs. Source: `RF_Surrogate_Room_Generalization.ipynb`. No fresh GPU execution is claimed.

# RF surrogate: geometry and TX/RX to predicted RF descriptors

Run on Colab T4. This experiment evaluates held-out configurations from a shared room family, not arbitrary new buildings. Reference answers come from Sionna. No RF simulation is required for predictor inference. Outputs and execution logs are saved to Drive. See PLAN.md and experiment.py for definitions. No results are claimed until evaluation completes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.check_output(['nvidia-smi'],text=True))

## Environment
Sionna runs GPU ray tracing; compact regression runs on the same host CPU. A GPU is not needed for the trained predictor.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, zipfile, json
SAVE=Path('/content/drive/MyDrive/RF_Surrogate_20260923')
SAVE.mkdir(exist_ok=True)
ROOT=Path('/content/rf_surrogate')
os.environ['RF_SURROGATE_OUT']=str(SAVE/'results')
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
subprocess.run([sys.executable,'-m','pip','install','-q','sionna-rt==1.2.1','scikit-learn==1.6.1','pandas','matplotlib','joblib'],check=True)

## Source package
Upload RF_Surrogate_Package.zip through the Files sidebar on the first run. Later runs restore the saved Drive package.

In [ ]:
PACKAGE=SAVE/'RF_Surrogate_Package.zip'
local_package=Path('/content/RF_Surrogate_Package.zip')
if local_package.exists(): shutil.copy2(local_package,PACKAGE)
assert PACKAGE.exists(),'Upload RF_Surrogate_Package.zip first.'
with zipfile.ZipFile(PACKAGE) as z: z.extractall('/content')
shutil.copytree(ROOT,SAVE/'source',dirs_exist_ok=True)
(SAVE/'logs').mkdir(exist_ok=True)
def run_stage(stage):
    with (SAVE/'logs'/(stage+'.log')).open('w') as log:
        process=subprocess.Popen([sys.executable,'-u',str(ROOT/'experiment.py'),stage],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=os.environ.copy())
        for line in process.stdout:
            print(line,end='');log.write(line);log.flush()
        assert process.wait()==0,'Stage failed: '+stage
run_stage('prepare')

## Physics and convergence checks
Test analytic free-space power/delay and double the ray count for two queries in each configuration. Stop on failure.

In [ ]:
run_stage('pilot')

## Reference dataset
Generate 160 random TX/RX pairs and five anchor queries per configuration. Save raw paths after each query. Resume rather than repeat.

In [ ]:
run_stage('generate')

## Train and select
Fit regressors and nearest-neighbour lookup using training configurations. Select the predictor using validation configurations only.

In [ ]:
run_stage('train')

## Final evaluation
Reload the saved model in a fresh process. Evaluate untouched test configurations, save predictions, compare baselines, measure latency, and construct documented 80-D embeddings.

In [ ]:
run_stage('evaluate')

## Fast full-output candidate
Exploratory follow-up: select Extra Trees using validation error and latency, retaining the original gradient-boosting results. No test labels enter this selection. The final speed claim uses a fresh matched benchmark below.

In [ ]:
subprocess.run([sys.executable,'-u',str(ROOT/'fast_model.py')],check=True,env=os.environ.copy())

## Same-runtime timing and reproducibility
Recompute 30 matched test queries across both configurations, verify their primary descriptors reproduce saved references, and time all 16 predicted outputs including feature extraction. One-time loading is separate.

In [ ]:
subprocess.run([sys.executable,'-u',str(ROOT/'benchmark.py')],check=True,env=os.environ.copy())

## Inspect and save final outputs
The report includes limitations. Results remain saved even if Colab disconnects.

In [ ]:
from IPython.display import display,Image
import pandas as pd,hashlib
R=SAVE/'results'
assert (R/'evaluation_complete.json').exists()
display(pd.read_csv(R/'test_metrics.csv'))
print((R/'timing.json').read_text())
display(Image(filename=str(R/'prediction_quality.png')))
display(Image(filename=str(R/'baseline_comparison.png')))
archive=shutil.make_archive('/content/RF_Surrogate_Results','zip',SAVE)
shutil.copy2(archive,SAVE.parent/'RF_Surrogate_Results.zip')
print('Saved results ZIP:',SAVE.parent/'RF_Surrogate_Results.zip')
print('SHA256',hashlib.sha256(Path(archive).read_bytes()).hexdigest())

## Read the fast model results
Higher speed does not establish acceptable accuracy. Compare each output to both baselines; this experiment only covers configurations of a shared room shell.

In [ ]:
display(pd.read_csv(R/'fast_model/test_metrics.csv'))
print((R/'same_runtime_benchmark.json').read_text())
display(pd.read_csv(R/'fast_model/embedding_quality.csv'))